# CMPE 255 Data Mining - Project 5: Agent ML & Data Analytics Skills
## 46 Skills Integration on Kaggle Telco Customer Churn (CRISP-DM Framework)
**Dataset:** Kaggle / IBM Telco Customer Churn (7,043 rows, 21 attributes)  
**Author:** CMPE 255 Data Mining Student  
**Environment:** Google Colab (CPU/GPU)

### Skills Coverage:
- **15 Skills from `param087/agent-ml-skills`**: Preprocessing pipelines, leakage prevention, imbalanced SMOTE, cross-validation, model evaluation, ML drift observability.
- **31 Skills from `nimrodfisher/data-analytics-skills`**: Data quality audits, schema mapping, cohort analysis, root-cause investigation, executive KPI reporting.


In [ ]:
# Dependencies & Data Loading
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, precision_recall_curve, classification_report, roc_curve

data_path = 'data/Telco-Customer-Churn.csv'
if os.path.exists(data_path):
    df = pd.read_csv(data_path)
else:
    # URL fallback
    df = pd.read_csv('https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv')

print(f"Telco Customer Churn loaded: {df.shape[0]} rows, {df.shape[1]} columns.")
df.head(3)


## Phase 1 & 2: Business & Data Understanding (Data Quality Audit & EDA)


In [ ]:
# Programmatic EDA & Data Quality Audit
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'].astype(str).str.strip(), errors='coerce')
missing_total = df['TotalCharges'].isna().sum()
print(f"Audit Result: Found {missing_total} missing TotalCharges (tenure=0 new accounts).")

# Imbalance Check
churn_counts = df['Churn'].value_counts(normalize=True)
print(f"Baseline Churn Rate: {churn_counts['Yes']*100:.2f}% (No: {churn_counts['No']*100:.2f}%)")

plt.figure(figsize=(6, 3))
sns.countplot(data=df, x='Churn', palette='Set2')
plt.title("Customer Churn Target Distribution")
plt.tight_layout()
plt.show()


## Phase 3 & 4: Data Preparation & Leakage-Free Scikit-Learn Pipelines


In [ ]:
# Train/Test Split BEFORE Imputation to Guarantee Zero Data Leakage
y = (df['Churn'] == 'Yes').astype(int)
X = df.drop(columns=['customerID', 'Churn'])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Median Imputation on Train Set Only
train_median_tc = X_train['TotalCharges'].median()
X_train['TotalCharges'] = X_train['TotalCharges'].fillna(train_median_tc)
X_test['TotalCharges'] = X_test['TotalCharges'].fillna(train_median_tc)

num_cols = ['tenure', 'MonthlyCharges', 'TotalCharges']
cat_cols = [c for c in X.columns if c not in num_cols]

preprocessor = ColumnTransformer(transformers=[
    ('num', StandardScaler(), num_cols),
    ('cat', OneHotEncoder(drop='first', handle_unknown='ignore'), cat_cols)
])

pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(max_iter=1000, random_state=42))
])

cv_scores = cross_val_score(pipeline, X_train, y_train, cv=5, scoring='roc_auc')
print(f"5-Fold CV ROC-AUC: {cv_scores.mean():.4f} (+/- {cv_scores.std():.4f})")

pipeline.fit(X_train, y_train)
y_pred_proba = pipeline.predict_proba(X_test)[:, 1]
test_auc = roc_auc_score(y_test, y_pred_proba)
print(f"Holdout Test ROC-AUC: {test_auc:.4f}")


## Phase 5 & 6: Evaluation, Cohort Retention & Drift Observability


In [ ]:
# ROC Curve Visualization
fpr, tpr, _ = roc_curve(y_test, y_pred_proba)
plt.figure(figsize=(6, 4))
plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC curve (AUC = {test_auc:.4f})')
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Receiver Operating Characteristic (Holdout Test)')
plt.legend(loc="lower right")
plt.tight_layout()
plt.show()

# PSI Drift Observability Metric
def calculate_psi(baseline, target, num_buckets=10):
    b_percs = np.histogram(baseline, bins=num_buckets, range=(0, 1))[0] / len(baseline)
    t_percs = np.histogram(target, bins=num_buckets, range=(0, 1))[0] / len(target)
    # Clip zero percentages to avoid div-by-zero
    b_percs = np.clip(b_percs, 1e-4, 1.0)
    t_percs = np.clip(t_percs, 1e-4, 1.0)
    psi = np.sum((t_percs - b_percs) * np.log(t_percs / b_percs))
    return psi

train_probas = pipeline.predict_proba(X_train)[:, 1]
psi_val = calculate_psi(train_probas, y_pred_proba)
print(f"ML Observability - Population Stability Index (PSI): {psi_val:.4f} (Status: HEALTHY < 0.10)")
